# Parquet i Arrow — format wymiany danych między Pandas, Polars, DuckDB i SQL Server

Parquet nie jest "jeszcze jednym formatem pliku" obok CSV — to kolumnowy
format binarny ze schematem, kompresją i metadanymi statystycznymi
wbudowanymi w plik, zaprojektowany właśnie pod wymianę danych między
różnymi silnikami. Ten notebook pokazuje dlaczego to ma znaczenie
praktyczne, nie tylko teoretyczne — na realnych pomiarach rozmiaru i czasu.

**Wymagania:** tylko `pyarrow` (Pandas/Polars/DuckDB używają go pod spodem
do obsługi Parquet — żadnych dodatkowych bibliotek do kompresji nie
potrzeba, wszystkie codeki są wbudowane).

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

rng = np.random.default_rng(42)

## 1. CSV vs Parquet — konkretna różnica, nie tylko deklaracja

Zamiast zaczynać od teorii, zmierzmy: ten sam zbiór danych, zapisany jako
CSV i jako Parquet — rozmiar pliku i czas odczytu.

In [2]:
n = 500_000
df_demo = pd.DataFrame({
    "id": np.arange(n),
    "kategoria": rng.choice(["A", "B", "C", "D"], n),          # niska kardynalność — dobrze się kompresuje
    "wartosc": rng.gamma(2, 150, n).round(2),
    "data": pd.date_range("2020-01-01", periods=n, freq="min"),
})

df_demo.to_csv("dane.csv", index=False)
df_demo.to_parquet("dane.parquet", index=False)   # domyślna kompresja: snappy

rozmiar_csv = Path("dane.csv").stat().st_size / 1024 / 1024
rozmiar_parquet = Path("dane.parquet").stat().st_size / 1024 / 1024

print(f"CSV:     {rozmiar_csv:.1f} MB")
print(f"Parquet: {rozmiar_parquet:.1f} MB  ({rozmiar_csv / rozmiar_parquet:.1f}x mniejszy)")

CSV:     16.9 MB
Parquet: 6.8 MB  (2.5x mniejszy)


In [3]:
start = time.perf_counter()
_ = pd.read_csv("dane.csv")
czas_csv = time.perf_counter() - start

start = time.perf_counter()
_ = pd.read_parquet("dane.parquet")
czas_parquet = time.perf_counter() - start

print(f"Odczyt CSV:     {czas_csv:.3f}s")
print(f"Odczyt Parquet: {czas_parquet:.3f}s  ({czas_csv / czas_parquet:.1f}x szybciej)")

Odczyt CSV:     0.464s
Odczyt Parquet: 0.146s  (3.2x szybciej)


**Dlaczego tak duża różnica:** CSV to tekst — każda liczba musi być
sparsowana ze stringa przy każdym odczycie, każdy wiersz to osobna linia
bez wspólnej struktury binarnej. Parquet trzyma dane **kolumnowo** (wartości
tej samej kolumny leżą obok siebie), z ustalonym typem binarnym i
kompresją per-kolumna — parsowanie tekstu znika, a podobne wartości obok
siebie kompresują się o wiele lepiej (stąd też różnica rozmiaru).

## 2. Schemat i typy — Parquet PAMIĘTA, CSV zgaduje

CSV nie przechowuje informacji o typach — przy każdym wczytaniu biblioteka
zgaduje (czy kolumna to `int`, `float`, `string`, `date`?) na podstawie
próbki wierszy, co bywa zawodne (kolumna wygląda na int, ale gdzieś w
środku ma tekst — cała kolumna staje się `object`). Parquet zapisuje
schemat **w pliku** — odczyt jest deterministyczny.

In [4]:
# problem z CSV: kolumna dat wczytuje się jako zwykły string, nie datetime — trzeba jawnie sparsować
df_z_csv = pd.read_csv("dane.csv")
print("Typ kolumny 'data' po wczytaniu z CSV:", df_z_csv["data"].dtype)   # object (string), NIE datetime

df_z_parquet = pd.read_parquet("dane.parquet")
print("Typ kolumny 'data' po wczytaniu z Parquet:", df_z_parquet["data"].dtype)   # datetime64 — zachowany automatycznie

Typ kolumny 'data' po wczytaniu z CSV: str
Typ kolumny 'data' po wczytaniu z Parquet: datetime64[us]


In [5]:
# schemat pliku Parquet można sprawdzić BEZ wczytywania danych do pamięci
metadane = pq.read_metadata("dane.parquet")
print(f"Liczba wierszy: {metadane.num_rows}")
print(f"Liczba grup wierszy (row groups): {metadane.num_row_groups}")
print()
print(pq.read_schema("dane.parquet"))

Liczba wierszy: 500000
Liczba grup wierszy (row groups): 1

id: int64
kategoria: large_string
wartosc: double
data: timestamp[us]
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 540


To, że schemat i podstawowe statystyki (liczba wierszy, zakresy wartości
per kolumna) są dostępne **bez wczytywania danych**, jest fundamentem
wydajności opisanej w sekcjach 4-5 — silnik może podjąć decyzje "czy w
ogóle warto czytać ten fragment pliku", zanim cokolwiek odczyta.

## 3. Kompresja — który codec wybrać

| Codec | Szybkość | Stopień kompresji | Kiedy wybrać |
|---|---|---|---|
| `snappy` | bardzo szybki | umiarkowany | **domyślny, dobry wybór w większości przypadków** — priorytet szybkości odczytu/zapisu |
| `gzip` | wolniejszy | lepszy niż snappy | gdy rozmiar pliku (koszt przechowywania/transferu) ważniejszy niż szybkość |
| `zstd` | szybki, konfigurowalny poziom | zwykle najlepszy kompromis | dobry domyślny wybór, jeśli chcesz lepszą kompresję niż snappy bez dużej straty szybkości |
| `lz4` | najszybszy | najsłabszy | priorytet absolutnej szybkości, rozmiar nieistotny |
| brak (`None`) | najszybszy zapis | brak kompresji | dane tymczasowe, gdzie i tak zaraz kasujesz plik |

In [6]:
for codec in ["snappy", "gzip", "zstd", "lz4", None]:
    nazwa_pliku = f"dane_{codec or 'brak'}.parquet"
    start = time.perf_counter()
    df_demo.to_parquet(nazwa_pliku, compression=codec, index=False)
    czas_zapisu = time.perf_counter() - start
    rozmiar = Path(nazwa_pliku).stat().st_size / 1024 / 1024
    print(f"{str(codec):8s}: {rozmiar:6.1f} MB, zapis {czas_zapisu:.3f}s")
    Path(nazwa_pliku).unlink()   # sprzątanie od razu, to tylko demonstracja

snappy  :    6.8 MB, zapis 0.167s


gzip    :    4.3 MB, zapis 7.681s
zstd    :    4.6 MB, zapis 0.137s


lz4     :    6.8 MB, zapis 0.100s
None    :    9.9 MB, zapis 0.090s


## 4. Odczyt tylko potrzebnych kolumn — column pruning

Skoro Parquet jest kolumnowy, odczyt **wybranych kolumn** czyta z dysku
TYLKO te kolumny — reszta pliku w ogóle nie jest dotykana. Przy szerokich
tabelach (dziesiątki kolumn), gdzie potrzebujesz kilku, to ogromna różnica
względem CSV (który musi sparsować każdy wiersz w całości, nawet jeśli
potem odrzucisz większość kolumn).

In [7]:
start = time.perf_counter()
_ = pd.read_parquet("dane.parquet", columns=["id", "wartosc"])   # tylko 2 z 4 kolumn
czas_2_kolumny = time.perf_counter() - start

start = time.perf_counter()
_ = pd.read_parquet("dane.parquet")   # wszystkie kolumny
czas_wszystkie = time.perf_counter() - start

print(f"Odczyt 2 z 4 kolumn:    {czas_2_kolumny:.3f}s")
print(f"Odczyt wszystkich kolumn: {czas_wszystkie:.3f}s")

Odczyt 2 z 4 kolumn:    0.038s
Odczyt wszystkich kolumn: 0.045s


Przy tylko 4 kolumnach różnica jest skromna — efekt column pruning rośnie
wraz z liczbą kolumn w tabeli źródłowej (przy tabeli z 50 kolumnami, gdzie
potrzebujesz 3, różnica bywa rzędu wielkości).

## 5. Partycjonowanie — pomijanie całych plików, nie tylko kolumn

Zapis "partycjonowany" dzieli dane na osobne pliki/foldery wg wartości
kolumny (typowo: data) — odczyt z filtrem na tej kolumnie **pomija całe
pliki**, nie tylko kolumny. To technika Hive-style, rozumiana przez
Pandas/Polars/DuckDB/Spark/SQL Server PolyBase jednolicie.

In [8]:
df_demo["rok"] = df_demo["data"].dt.year
df_demo["miesiac"] = df_demo["data"].dt.month

# partition_cols -> osobny podfolder na każdą kombinację (rok, miesiąc): dane_partycjonowane/rok=2020/miesiac=1/...
df_demo.to_parquet("dane_partycjonowane", partition_cols=["rok", "miesiac"], index=False)

# struktura utworzonych folderów (fragment)
foldery = sorted(Path("dane_partycjonowane").glob("rok=*/miesiac=*"))
print(f"Utworzono {len(foldery)} partycji, np.:")
for f in foldery[:3]:
    print(" ", f)

Utworzono 12 partycji, np.:
  dane_partycjonowane/rok=2020/miesiac=1
  dane_partycjonowane/rok=2020/miesiac=10
  dane_partycjonowane/rok=2020/miesiac=11


In [9]:
# odczyt z filtrem na kolumnie partycjonującej -> silnik czyta TYLKO pasujące foldery/pliki
start = time.perf_counter()
tylko_styczen = pd.read_parquet("dane_partycjonowane", filters=[("miesiac", "==", 1)])
czas_z_filtrem = time.perf_counter() - start

start = time.perf_counter()
wszystko = pd.read_parquet("dane_partycjonowane")
czas_bez_filtra = time.perf_counter() - start

print(f"Odczyt z filtrem (1 miesiąc):  {czas_z_filtrem:.3f}s, {len(tylko_styczen)} wierszy")
print(f"Odczyt bez filtra (12 miesięcy): {czas_bez_filtra:.3f}s, {len(wszystko)} wierszy")

Odczyt z filtrem (1 miesiąc):  0.018s, 44640 wierszy
Odczyt bez filtra (12 miesięcy): 0.107s, 500000 wierszy


`filters=[("miesiac", "==", 1)]` to tzw. **predicate pushdown** — filtr
jest stosowany na poziomie "które pliki w ogóle otworzyć", zanim
cokolwiek trafi do pamięci jako DataFrame. Ten sam mechanizm (tylko
składniowo inaczej wyrażony) działa w Polars (`pl.scan_parquet` + `.filter()`,
sekcja 7) i DuckDB (zwykłe `WHERE` w SQL na pliku Parquet).

**Kiedy partycjonować:** kolumna o rozsądnej liczbie unikalnych wartości
(dni/miesiące/regiony — dziesiątki/setki, nie miliony) i gdy realnie
filtrujesz po niej w większości zapytań. Partycjonowanie po kolumnie o
wysokiej kardynalności (np. `klient_id` z 2 mln unikalnych wartości)
tworzy setki tysięcy malutkich plików — pogarsza wydajność zamiast pomagać.

## 6. Ten sam plik, trzy silniki — Pandas, Polars, DuckDB

To jest sedno "formatu wymiany" — jeden plik Parquet, czytany bez
konwersji przez wszystkie trzy narzędzia z Twojego stacku.

In [10]:
# Pandas
df_pandas = pd.read_parquet("dane.parquet")

# Polars
df_polars = pl.read_parquet("dane.parquet")

# DuckDB — SQL bezpośrednio na pliku, bez wcześniejszego wczytywania do DataFrame
wynik_duckdb = duckdb.sql("SELECT kategoria, COUNT(*) AS n, AVG(wartosc) AS srednia FROM 'dane.parquet' GROUP BY kategoria").pl()

print(type(df_pandas), df_pandas.shape)
print(type(df_polars), df_polars.shape)
print(wynik_duckdb)

<class 'pandas.DataFrame'> (500000, 4)
<class 'polars.dataframe.frame.DataFrame'> (500000, 4)
shape: (4, 3)
┌───────────┬────────┬────────────┐
│ kategoria ┆ n      ┆ srednia    │
│ ---       ┆ ---    ┆ ---        │
│ str       ┆ i64    ┆ f64        │
╞═══════════╪════════╪════════════╡
│ A         ┆ 125272 ┆ 301.554876 │
│ D         ┆ 124911 ┆ 299.521835 │
│ C         ┆ 125318 ┆ 300.324506 │
│ B         ┆ 124499 ┆ 299.223578 │
└───────────┴────────┴────────────┘


DuckDB nawet nie musi "wczytać" pliku w sensie Python-owym — silnik SQL
czyta plik Parquet bezpośrednio z dysku, stosując column pruning i
predicate pushdown automatycznie na podstawie treści zapytania SQL (np.
`WHERE`/`GROUP BY` w tym zapytaniu). To zwykle najszybsza opcja do
transformacji "prosto z pliku", bez pośredniego kroku wczytania do
DataFrame.

## 7. Lazy evaluation w Polars — jeszcze więcej optymalizacji "za darmo"

`pl.read_parquet` wczytuje dane od razu (eager). `pl.scan_parquet` buduje
**plan zapytania**, który wykonuje się dopiero na `.collect()` — Polars
może wtedy połączyć column pruning, predicate pushdown i inne optymalizacje
w jeden przebieg, zamiast wczytywać wszystko, a dopiero potem filtrować.

In [11]:
plan = (
    pl.scan_parquet("dane.parquet")
    .select(["kategoria", "wartosc"])        # column pruning — Polars wie, że nie potrzebuje 'id'/'data'
    .filter(pl.col("wartosc") > 500)          # predicate pushdown — filtr zastosowany jak najwcześniej
    .group_by("kategoria")
    .agg(pl.col("wartosc").mean().alias("srednia"))
)

print(plan.explain())   # plan wykonania — widać "PROJECT" i predicate zanim cokolwiek się wykona

AGGREGATE[maintain_order: false]
  [col("wartosc").mean().alias("srednia")] BY [col("kategoria")]
  FROM
  Parquet SCAN [dane.parquet]
  PROJECT 2/4 COLUMNS
  SELECTION: col("wartosc") > 500.0
  ESTIMATED ROWS: 500000


In [12]:
wynik = plan.collect()   # dopiero teraz plan faktycznie się wykonuje
wynik

kategoria,srednia
str,f64
"""B""",683.573633
"""A""",682.736149
"""D""",683.681039
"""C""",686.415945


Zasada praktyczna: przy pracy z plikami Parquet w Polars (zwłaszcza dużymi
albo wieloma naraz) domyślnie sięgaj po `scan_parquet` + łańcuch
`.select()/.filter()/.group_by()` + `.collect()` na końcu, zamiast
`read_parquet` + operacje na już wczytanym DataFrame — różnica rośnie z
rozmiarem danych i liczbą plików.

## 8. Wiele plików naraz — Parquet jako naturalny format "folderu danych"

Typowy wzorzec: codzienny eksport z SQL Server jako osobny plik Parquet,
wszystkie razem odczytywane jako jeden logiczny zbiór (glob pattern),
bez ręcznego `pd.concat([pd.read_parquet(p) for p in pliki])`.

In [13]:
for dzien in range(1, 4):
    df_dnia = df_demo.sample(1000, random_state=dzien)
    df_dnia.to_parquet(f"eksport_dzien_{dzien}.parquet", index=False)

# DuckDB i Polars czytają wzorzec z wieloma plikami jednym wywołaniem, bez ręcznego concat
polaczone_duckdb = duckdb.sql("SELECT COUNT(*) AS liczba_wierszy FROM 'eksport_dzien_*.parquet'").pl()
polaczone_polars = pl.scan_parquet("eksport_dzien_*.parquet").select(pl.len()).collect()

print("DuckDB:", polaczone_duckdb)
print("Polars:", polaczone_polars)

for dzien in range(1, 4):
    Path(f"eksport_dzien_{dzien}.parquet").unlink()

DuckDB: shape: (1, 1)
┌────────────────┐
│ liczba_wierszy │
│ ---            │
│ i64            │
╞════════════════╡
│ 3000           │
└────────────────┘
Polars: shape: (1, 1)
┌──────┐
│ len  │
│ ---  │
│ u32  │
╞══════╡
│ 3000 │
└──────┘


## 9. Eksport z SQL Server do Parquet — praktyczny wzorzec

Złożenie z notebookiem o `mssql_python`: zamiast zapisywać wyniki zapytań
SQL Server do CSV (wolne, bez typów, duże pliki), zapisuj jako Parquet —
bezpośrednio z DataFrame Polars/Pandas, które i tak już budujesz przy
odczycie.

```python
from mssql_python import connect
import polars as pl

with connect(CONNECTION_STRING) as conn:
    df = pl.read_database("SELECT * FROM Sales.Zamowienia WHERE DataZamowienia >= '2025-01-01'", conn)

df.write_parquet(
    "eksport_zamowienia.parquet",
    compression="zstd",   # dobry domyślny wybór (sekcja 3)
)
```

Dla dużych, regularnych eksportów (np. codzienny task w Airflow z
poprzedniego notebooka) — zapis partycjonowany po dacie (sekcja 5) daje
naturalny, przyrostowy archiwum: nowy dzień = nowy plik/partycja, bez
przepisywania całej historii.

## 10. Arrow — wspólny format W PAMIĘCI, nie tylko na dysku

Parquet to format **na dysku**. Arrow to format **w pamięci** — i to on
jest powodem, dla którego Pandas → Polars → DuckDB mogą wymieniać dane
**bez kopiowania** (zero-copy), jeśli tylko odpowiednio o to zadbasz.

In [14]:
# konwersja Pandas -> Arrow Table -> Polars, ZERO-COPY gdy typy danych są kompatybilne
tabela_arrow = pa.Table.from_pandas(df_pandas[["id", "wartosc"]])
df_polars_z_arrow = pl.from_arrow(tabela_arrow)

print(type(tabela_arrow))
print(type(df_polars_z_arrow))

<class 'pyarrow.lib.Table'>
<class 'polars.dataframe.frame.DataFrame'>


In [15]:
# DuckDB potrafi też odpytać Arrow Table bezpośrednio, bez konwersji do żadnego DataFrame
wynik_z_arrow = duckdb.sql("SELECT AVG(wartosc) AS srednia FROM tabela_arrow").fetchone()
print(f"Średnia policzona przez DuckDB na Arrow Table: {wynik_z_arrow[0]:.2f}")

Średnia policzona przez DuckDB na Arrow Table: 300.16


Praktyczna korzyść: w pipeline'ie łączącym kilka silników (np. wczytanie
przez `mssql_python`/Polars, transformacja w DuckDB, coś jeszcze w Pandas
dla konkretnej biblioteki, która wymaga Pandas) **nie musisz** za każdym
razem przechodzić przez zapis/odczyt pliku ani przez kosztowną konwersję
kopiującą całe dane — Arrow jest wspólnym mianownikiem "w locie".

## Podsumowanie

| Pytanie | Odpowiedź |
|---|---|
| CSV czy Parquet do zapisu wyników pośrednich? | Parquet — mniejszy plik, szybszy odczyt, zachowane typy (sekcja 1-2) |
| Który codec kompresji? | `snappy` (domyślny, szybki) albo `zstd` (lepsza kompresja, wciąż szybki) |
| Potrzebuję tylko kilku kolumn z szerokiej tabeli? | Parquet czyta tylko te kolumny automatycznie (column pruning) |
| Duży zbiór, często filtrowany po dacie/regionie? | Partycjonuj po tej kolumnie (`partition_cols`) — ale nie po kolumnie wysokiej kardynalności |
| Polars na dużym pliku/wielu plikach? | `scan_parquet` + łańcuch operacji + `.collect()` na końcu, nie `read_parquet` od razu |
| Transformacja "prosto z pliku", bez pośredniego DataFrame? | DuckDB SQL bezpośrednio na ścieżce/wzorcu pliku |
| Wymiana danych między Pandas/Polars/DuckDB w jednym procesie? | Arrow (`pa.Table`) jako wspólny format w pamięci, bez kopiowania |
| Eksport z SQL Server do archiwum/pipeline'u? | `mssql_python` → Polars/Pandas → `write_parquet`/`to_parquet`, opcjonalnie partycjonowany |